### What we are building



A state machine based secure authentication pipeline

Flow:

<pre>
Detect Person
    ↓
Recognize Identity
    ↓
Check Liveness
    ↓
Ask: Blink
    ↓
Ask: Turn Left
    ↓
Ask: Turn Right
    ↓
Authorize
</pre>

In [10]:
import time
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
import os
import torch.nn.functional as F
import torch

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

mtcnn = MTCNN(image_size=160, margin=0, device=device, keep_all=True)

resnet = InceptionResnetV1(pretrained="vggface2").eval().to(device)

In [12]:
database = {}

for person in os.listdir('faces/single'):
    person_embeddings = []
    
    for file in os.listdir(f'faces/single/{person}'):
        img = Image.open(os.path.join(f'faces/single/{person}', file))
        
        face = mtcnn(img)
        
        if face is None:
            continue
        
        # If multiple faces are detected take first one:
        if face.ndim == 4:
            face = face[0].unsqueeze(0)
        
        face = face.to(device)
        
        with torch.no_grad():
            embedding = resnet(face)
            
        # Normalize embedding
        embedding /= embedding.norm(dim=1, keepdim=True)
        
        person_embeddings.append(embedding)
        
    if len(person_embeddings) > 0:
        # Average embeddings
        database[person] = torch.mean(torch.cat(person_embeddings), dim=0, keepdim=True)
            
print("Database Loaded: ", list(database.keys()))

Database Loaded:  ['abhishek_bachchan', 'amitabh_bachchan', 'chirag', 'jaya_bachchan', 'shail']


In [13]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

In [14]:
def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def calculate_ear(eye_points):
    A = euclidean(eye_points[1], eye_points[5])
    B = euclidean(eye_points[2], eye_points[4])
    C = euclidean(eye_points[0], eye_points[3])

    ear = (A + B) / (2.0 * C)
    return ear

In [19]:
AUTH_DURATION = 15
EAR_THRESHOLD = 0.2
BLINK_FRAMES = 1
HEAD_THRESHOLD = 40

active_sessions = {}          # { person: expiry_timestamp }
current_auth_target = None    # Only one person at a time

blink_counter = 0
auth_state = "IDLE"

In [20]:
def recognize_faces(frame, database, threshold=0.75):

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(frame_rgb)

    boxes, _ = mtcnn.detect(img)

    if boxes is None:
        time.sleep(0.25)
        return frame, []

    faces = mtcnn(img)

    if faces is None:
        time.sleep(0.25)
        return frame, []

    if faces.ndim == 3:
        faces = faces.unsqueeze(0)

    faces = faces.to(device)

    with torch.no_grad():
        embeddings = resnet(faces)

    embeddings /= embeddings.norm(dim=1, keepdim=True)

    results = []

    for i, query_embedding in enumerate(embeddings):

        query_embedding = query_embedding.unsqueeze(0)

        best_match = None
        max_similarity = -1

        for name, db_embedding in database.items():

            similarity = F.cosine_similarity(
                query_embedding,
                db_embedding
            ).item()

            if similarity > max_similarity:
                max_similarity = similarity
                best_match = name

        if max_similarity > threshold:
            person_name = best_match
        else:
            person_name = None

        results.append((person_name, boxes[i]))

    return frame, results


In [21]:
def run_liveness(frame, person_name):

    global blink_counter, auth_state, current_auth_target

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=frame_rgb
    )

    timestamp = int(time.time() * 1000)
    result = landmarker.detect_for_video(mp_image, timestamp)

    if not result.face_landmarks:
        return frame, False

    h, w, _ = frame.shape
    landmarks = result.face_landmarks[0]

    # ---------- HEAD ----------
    nose = landmarks[1]
    nose_x = int(nose.x * w)
    center_x = w // 2
    offset = nose_x - center_x

    if offset > HEAD_THRESHOLD:
        head_dir = "LEFT"
    elif offset < -HEAD_THRESHOLD:
        head_dir = "RIGHT"
    else:
        head_dir = "CENTER"

    # ---------- EYES ----------
    left_eye_indices = [33, 160, 158, 133, 153, 144]
    right_eye_indices = [362, 385, 387, 263, 373, 380]

    left_pts = []
    right_pts = []

    for idx in left_eye_indices:
        x = int(landmarks[idx].x * w)
        y = int(landmarks[idx].y * h)
        left_pts.append((x, y))
        cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

    for idx in right_eye_indices:
        x = int(landmarks[idx].x * w)
        y = int(landmarks[idx].y * h)
        right_pts.append((x, y))
        cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

    ear_left = calculate_ear(left_pts)
    ear_right = calculate_ear(right_pts)
    ear = (ear_left + ear_right) / 2

    blink_detected = False

    if ear < EAR_THRESHOLD:
        blink_counter += 1
    else:
        if blink_counter >= BLINK_FRAMES:
            blink_detected = True
        blink_counter = 0

    # ---------- STATE MACHINE ----------

    if auth_state == "IDLE":
        auth_state = "BLINK"
        blink_counter = 0

    elif auth_state == "BLINK":
        cv2.putText(frame, "Please Blink", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 1,
                    (0,255,0), 2)

        if blink_detected:
            auth_state = "LEFT"

    elif auth_state == "LEFT":
        cv2.putText(frame, "Turn Head Left", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 1,
                    (0,255,0), 2)

        if head_dir == "LEFT":
            auth_state = "RIGHT"

    elif auth_state == "RIGHT":
        cv2.putText(frame, "Turn Head Right", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 1,
                    (0,255,0), 2)

        if head_dir == "RIGHT":
            auth_state = "AUTHORIZED"

    elif auth_state == "AUTHORIZED":
        active_sessions[person_name] = time.time() + AUTH_DURATION
        current_auth_target = None
        auth_state = "IDLE"
        return frame, True

    return frame, False


In [22]:
cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()
    frame = cv2.resize(frame, (640,480))

    if not ret:
        break

    frame, detections = recognize_faces(frame, database)

    current_time = time.time()

    for person_name, box in detections:

        x1, y1, x2, y2 = map(int, box)

        if person_name:

            # Already authorized?
            if person_name in active_sessions:

                expiry = active_sessions[person_name]

                if current_time < expiry:
                    remaining = int(expiry - current_time)
                    label = f"{person_name} ({remaining}s)"
                    color = (0,255,0)
                else:
                    del active_sessions[person_name]
                    label = f"{person_name} - Reauth"
                    color = (0,0,255)

            else:
                label = f"{person_name} - Verify"
                color = (0,255,255)

                # Start authentication if none running
                if current_auth_target is None:
                    current_auth_target = person_name

        else:
            label = "Unknown"
            color = (0,0,255)

        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
        cv2.putText(frame, label, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, color, 2)

    # Run liveness only for current target
    if current_auth_target:
        frame, _ = run_liveness(frame, current_auth_target)

    cv2.imshow("Multi-Person Secure System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()